# 09. Structured & Record Arrays (5+ Years Interview Guide)
Exhaustive revision guide to heterogeneous composite dtypes, named column access, np.recarray dot-notation, and struct alignment on transaction data.

### Key 5-Year Interview Concepts Covered:
- **Dtype Construction (`np.dtype`)**: Defining multi-type fields using `np.dtype([('name', 'U10'), ('age', 'i4'), ('weight', 'f4')])`.
- **Field Access**: Referencing columns by field name (`structured_arr['age']`).
- **Record Arrays (`np.recarray`)**: Enabling dot-notation attribute access (`arr.age`).
- **Memory Alignment (`align=True`)**: Compiler padding bytes for C-struct hardware alignment.

This interactive revision guide loads and operates directly on `data/raw_transactions.csv` using dedicated cells per method.

In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from data/raw_transactions.csv (14262 clean aligned rows):
- amounts array: shape (14262,), dtype float64
- fraud_flags array: shape (14262,), dtype int8
- account_ages array: shape (14262,), dtype float32


### Dtype Construction with `np.dtype`
**Explanation**: Constructs a C-struct array holding transaction ID, amount, and fraud flag in a single contiguous binary buffer.

**Syntax**: `np.dtype([('tx_id', 'U10'), ('amount', 'f8'), ('is_fraud', 'i1')])`

In [2]:
tx_struct_dtype = np.dtype([
    ('tx_id', 'U12'),
    ('amount', np.float64),
    ('is_fraud', np.int8)
])
struct_records = np.array([
    (clean_raw['transaction_id'][i], clean_raw['transaction_amount'][i], clean_raw['is_fraud'][i])
    for i in range(5)
], dtype=tx_struct_dtype)
print('Structured Transactions Array:\n', struct_records)

Structured Transactions Array:
 [('TX110686', 1216.33, 0) ('TX107170',  324.99, 0)
 ('TX108328',  136.66, 0) ('TX108563',  124.21, 0)
 ('TX107002', 1284.68, 0)]


### Field Access: `arr['field']`
**Explanation**: Extracts `amount` and `is_fraud` fields as zero-copy views.

**Syntax**: `struct_records['amount']`

In [3]:
print('Amount Field View:', struct_records['amount'])
print('Fraud Field View:', struct_records['is_fraud'])

Amount Field View: [1216.33  324.99  136.66  124.21 1284.68]
Fraud Field View: [0 0 0 0 0]


### Record Arrays with Dot Notation (`np.recarray`)
**Explanation**: Enables dot-notation attribute access (`rec.amount`) over structured memory.

**Syntax**: `rec = struct_records.view(np.recarray)`

In [4]:
rec_tx = struct_records.view(np.recarray)
print('Dot Notation Access (rec_tx.tx_id):', rec_tx.tx_id)
print('Dot Notation Access (rec_tx.amount):', rec_tx.amount)

Dot Notation Access (rec_tx.tx_id): ['TX110686' 'TX107170' 'TX108328' 'TX108563' 'TX107002']
Dot Notation Access (rec_tx.amount): [1216.33  324.99  136.66  124.21 1284.68]


### Memory Alignment with `align=True`
**Explanation**: Pads composite C-structs for 64-bit hardware alignment.

**Syntax**: `np.dtype([('flag', 'i1'), ('amt', 'f8')], align=True)`

In [5]:
unaligned_dt = np.dtype([('flag', 'i1'), ('amt', 'f8')])
aligned_dt = np.dtype([('flag', 'i1'), ('amt', 'f8')], align=True)
print('Unaligned itemsize:', unaligned_dt.itemsize, 'bytes')
print('Aligned itemsize (padded):', aligned_dt.itemsize, 'bytes')

Unaligned itemsize: 9 bytes
Aligned itemsize (padded): 16 bytes


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Filtering Structured Transaction Records
**Explanation**: Filter structured transaction records where `amount > 200` and `is_fraud == 0`.

**Syntax**: `struct_records[(struct_records['amount'] > 200) & (struct_records['is_fraud'] == 0)]`

In [6]:
filtered_records = struct_records[(struct_records['amount'] > 100) & (struct_records['is_fraud'] == 0)]
print('Filtered Valid Records:\n', filtered_records)

Filtered Valid Records:
 [('TX110686', 1216.33, 0) ('TX107170',  324.99, 0)
 ('TX108328',  136.66, 0) ('TX108563',  124.21, 0)
 ('TX107002', 1284.68, 0)]
